# 02 · Render storms — renders/s against the daemon event rate

A storm is the GUI re-rendering far faster than anything upstream changed. The
attribution question is therefore always comparative: **did the daemon get
busier, or did the GUI start spinning on its own?**

Prior art (`docs/pending-bugs.md`, render-pipeline): a post-restart root at
54–64 renders/s against a calm baseline of 0.7–1.2/s, one core pinned for ~9
minutes, and a **flat daemon event rate across the whole episode** — driven by
one state write per render that changed no watched field.

⛔ `render/*` is the one category on the **cpu** clock: `duration_ms` is CPU-ms
consumed over `interval_ms`, not a latency. Cores = `duration_ms / interval_ms`.
Records written before the payload mirror was fixed carry no interval and are
counted as unreadable rather than as zero.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) != "notebooks" else os.getcwd()))
sys.path.insert(0, os.path.abspath("."))
import ytrace_helpers as H

WINDOW = os.environ.get("YGG_NOTEBOOK_WINDOW", "30m")
HOST = H.GUI_HOST
H.describe_source(HOST, WINDOW)

In [ ]:
CALM_RPS, STORM_RPS = 1.2, 20.0
BUCKET_MS = 10_000

render_rate = [r for r in H.tail(HOST, since=WINDOW, category="ui")
               if r.get("name") == "app_render_rate"]
render_rate += [r for r in H.tail(HOST, since=WINDOW, category="perf")
                if r.get("name") == "app_render_rate"]
daemon_evts = H.tail(HOST, since=WINDOW, category="request")
storms = [r for r in H.tail(HOST, since=WINDOW, category="render") if r.get("name") == "storm"]

print(f"app_render_rate samples : {len(render_rate)}")
print(f"daemon request events   : {len(daemon_evts)}")
print(f"render/storm incidents  : {len(storms)}")

In [ ]:
# Two series on ONE bucket grid, so a rise in the first can be read against
# the second. Empty buckets are kept as zero — a storm is only visible against
# the calm around it.
gui_series = H.rate_series(render_rate, BUCKET_MS)
dmn_series = H.rate_series(daemon_evts, BUCKET_MS)

def report(label, series, unit):
    if not series:
        print(f"{label}: (no data)")
        return None
    vals = [v for _, v in series]
    print(f"{label}: {H.sparkline(vals)}  max={max(vals):.2f} {unit}")
    print(f"{'':>{len(label)}}  {H.ts(series[0][0])} .. {H.ts(series[-1][0])}")
    return vals

gui_vals = report("GUI render events/s   ", gui_series, "ev/s")
dmn_vals = report("daemon request ev/s   ", dmn_series, "ev/s")

In [ ]:
# The reported rate lives in the probe payload; the event rate above is only a
# proxy for how often the probe fired.
reported = []
for r in render_rate:
    p = r.get("payload") or {}
    if isinstance(p, dict):
        for key in ("renders_per_sec", "rate", "renders_per_second", "value"):
            if isinstance(p.get(key), (int, float)):
                reported.append(float(p[key]))
                break
print("reported renders/s:", H.percentiles(reported) if reported else "(payload carries no rate field)")
if reported:
    print("series:", H.sparkline(reported))

In [ ]:
# Render COST, in cores. This is the cpu-clock category.
render_rows = H.tail(HOST, since=WINDOW, category="render")
by_role, unreadable = {}, 0
for r in render_rows:
    cf = H.core_fraction(r)
    if cf is None:
        if r.get("duration_ms") is not None:
            unreadable += 1
        continue
    by_role.setdefault(r.get("name", "?"), []).append(cf)

rows = []
for role, vals in sorted(by_role.items(), key=lambda kv: -H.percentiles(kv[1]).get("p50", 0)):
    st = H.percentiles(vals)
    rows.append({"role": role, "n": st["n"], "p50_cores": st.get("p50"),
                 "p95_cores": st.get("p95"), "max_cores": st.get("max")})
print(H.table(rows, ["role", "n", "p50_cores", "p95_cores", "max_cores"]) if rows else "(no readable render samples)")
if unreadable:
    print(f"\n{unreadable} cpu-clock spans had no interval_ms and are UNREADABLE, not zero "
          "(written before the payload mirror was fixed).")
total_p50 = sum(H.percentiles(v).get("p50", 0.0) for v in by_role.values()) if by_role else None
print(f"\ntotal render tree p50: {total_p50:.2f} cores" if total_p50 is not None else "\ntotal: unknown")

In [ ]:
# Storm onset attribution — asked ONLY when there is a storm to attribute.
# Naming an "onset" during a calm window is noise dressed as analysis: the peak
# of a flat series is just its largest sample.
onset = None
peak_rate = max(reported) if reported else (max(v for _, v in gui_series) if gui_series else None)

if peak_rate is None or peak_rate < CALM_RPS:
    shown = "unknown" if peak_rate is None else f"{peak_rate:.2f}"
    print(f"GUI peak {shown} renders/s is within the calm band (<= {CALM_RPS}). "
          "No storm to attribute.")
else:
    peak_bucket, _ = max(gui_series, key=lambda kv: kv[1])
    dmn_at_peak = dict(dmn_series).get(peak_bucket, 0.0)
    dmn_median = H.percentiles([v for _, v in dmn_series]).get("p50", 0.0) if dmn_series else 0.0
    ratio = (dmn_at_peak / dmn_median) if dmn_median else None
    onset = {"bucket": peak_bucket, "gui_rate": peak_rate, "daemon_rate": dmn_at_peak,
             "daemon_median": dmn_median, "daemon_ratio": ratio}
    print(f"GUI peak {peak_rate:.2f} renders/s at {H.ts(peak_bucket)}")
    print(f"daemon at that moment {dmn_at_peak:.2f} ev/s vs median {dmn_median:.2f} ev/s"
          + (f" ({ratio:.2f}x)" if ratio else ""))
    print("\n=> " + ("daemon rose with it — upstream change, not a self-driven storm"
                     if ratio and ratio > 1.5 else
                     "daemon FLAT across the GUI peak — the signature of a self-driven storm"))

In [ ]:
RENDER_TOTAL_CORE_THRESHOLD = 0.70   # ytrace::diagnosis, kept in step deliberately

v = H.Verdict("Render storms")

# The RATE the probe REPORTS, not how often the probe fired. Those differ by the
# sampling period and confusing them understates a storm by ~an order of
# magnitude — the probe can emit once a minute while the GUI renders 60x/s.
rst = H.percentiles(reported)
v.check("renders/s (reported, p95)", rst.get("p95"),
        f"calm <= {CALM_RPS} /s, storm >= {STORM_RPS} /s",
        warn_over=CALM_RPS, fail_over=STORM_RPS, n=rst.get("n", 0))
v.check("renders/s (reported, max)", rst.get("max"), f"storm >= {STORM_RPS} /s",
        warn_over=STORM_RPS / 2, fail_over=STORM_RPS, n=rst.get("n", 0))
v.check("render tree cost (p50 cores)", total_p50,
        f"<= {RENDER_TOTAL_CORE_THRESHOLD} cores",
        warn_over=RENDER_TOTAL_CORE_THRESHOLD * 0.7, fail_over=RENDER_TOTAL_CORE_THRESHOLD)
v.check("durable render/storm incidents", float(len(storms)), "0", warn_over=0, fail_over=2)

if onset is None:
    v.note(H.PASS, "storm onset attribution", "no storm in this window — nothing to attribute")
elif onset["daemon_ratio"] is None:
    v.note(H.UNKNOWN, "storm onset attribution", "daemon series too sparse to compare")
else:
    v.note(H.WARN if onset["daemon_ratio"] < 1.5 else H.PASS, "storm onset attribution",
           f"daemon was {onset['daemon_ratio']:.2f}x its median at the GUI peak — "
           + ("flat upstream, so the GUI is driving itself" if onset["daemon_ratio"] < 1.5
              else "upstream got busier, so the GUI is following real work"))
if unreadable:
    v.note(H.WARN, "unreadable cpu spans",
           f"{unreadable} render spans carry no interval_ms — cost cannot be computed from them")
v.show()